# Create Non Coding Data Set
Andrew E. Davidson
aedaivds@ucsc.edu 02/10/25

Copyright (c) 2020-2023, Regents of the University of California All rights reserved. https://polyformproject.org/licenses/noncommercial/1.0.0

ref: /private/groups/kimlab/data/elife/README.md


1. Get the biomarkers from an old Create run  
    * Load the Elife 20241107/create/annotated_norm_counts.csv
    * extra the gene_id column and the biotype column

2. ~~Load the tempus/illumina/20241220/results/data/normalized_counts.csv~~
    * this does not have the biomarker columnm

3. load the elife count data
   * /private/groups/kimlab/data/elife/elife_all_norm_counts_2023-05-18.csv

4. join on gene_id


**data files** 
18 rows are miss
/private/groups/kimlab/aedavids/elife/createNonCodingDataSet.out/data/annotated_elife_all_norm_counts_2023-05-18.csv 

In [1]:
import ipynbname

# use display() to print an html version of a data frame
# useful if dataFrame output is not generated by last like of cell
from IPython.display import display

import joblib
import math
import numpy as np
import os
import pandas as pd
import pprint as pp
import sys

notebookName = ipynbname.name()
notebookPath = ipynbname.path()
notebookDir = os.path.dirname(notebookPath)

#outDir = f'{notebookDir}/{notebookName}.out'
outDir = f'/private/groups/kimlab/aedavids/elife/{notebookName}.out'
os.makedirs(outDir, exist_ok=True)
print(f'outDir:\n{outDir}')

dataOutDir = os.path.join(outDir, "data")
os.makedirs(dataOutDir, exist_ok=True)
print(f'\ndataOutDir ;\n{dataOutDir}')

import logging
#loglevel = "DEBUG"
#loglevel = "INFO"
loglevel = "WARN"
# logFMT = "%(asctime)s %(levelname)s [thr:%(threadName)s %(name)s %(funcName)s() line:%(lineno)s] [%(message)s]"
logFMT = "%(asctime)s %(levelname)s %(name)s %(funcName)s() line:%(lineno)s] [%(message)s]"
logging.basicConfig(format=logFMT, level=loglevel)    
logger = logging.getLogger(notebookName)

outDir:
/private/groups/kimlab/aedavids/elife/createNonCodingDataSet.out

dataOutDir ;
/private/groups/kimlab/aedavids/elife/createNonCodingDataSet.out/data


In [2]:
%%time
tempusPath = "/private/groups/kimlab/data/tempus/illumina/20241107/create/annotated_norm_counts.csv"
tempusCountsDF= pd.read_csv( tempusPath, index_col='gene_id' )

CPU times: user 313 ms, sys: 60 ms, total: 373 ms
Wall time: 829 ms


In [3]:
print( tempusCountsDF.shape )
tempusCountsDF.iloc[0:5, 0:4]

(76539, 37)


,gene_name,gene_biotype,SLDK3_T1_100_S1_L007,SLDK3_T1_100K_S2_L007
gene_id,,,,
(A)n,(A)n,Microsatellite,0.545009,1.987324
(AAA)n,(AAA)n,Microsatellite,0.000000,0.000000
(AAAAAAC)n,(AAAAAAC)n,Microsatellite,0.000000,0.000000
(AAAAAAG)n,(AAAAAAG)n,Microsatellite,0.000000,0.000000
(AAAAAAT)n,(AAAAAAT)n,Microsatellite,0.000000,0.000000


In [4]:
%%time
elifePath = "/private/groups/kimlab/data/elife/elife_all_norm_counts_2023-05-18.csv"
elifeCountsDF = pd.read_csv(elifePath, index_col='gene')

CPU times: user 2.27 s, sys: 239 ms, total: 2.51 s
Wall time: 5.46 s


In [5]:
print( elifeCountsDF.shape )
elifeCountsDF.iloc[0:5, 0:5]

(76555, 224)


,SRR14506659,SRR14506660,SRR14506661,SRR14506662,SRR14506663
gene,,,,,
(A)n,201.672053,110.450773,3722.776395,1394.605651,2843.47373
(AAA)n,0.000000,0.000000,0.000000,0.000000,0.00000
(AAAAAAC)n,0.000000,0.000000,0.000000,0.000000,0.00000
(AAAAAAG)n,0.000000,0.000000,0.000000,0.000000,0.00000
(AAAAAAT)n,0.000000,0.000000,0.000000,0.000000,0.00000


In [6]:
print(f'the number of rows is different elifeCountsDF.shape[0] - tempusCountsDF.shape[0] = {elifeCountsDF.shape[0] - tempusCountsDF.shape[0]}' )

the number of rows is different elifeCountsDF.shape[0] - tempusCountsDF.shape[0] = 16


In [15]:
%%time

elifeWithBiomarkerDF = pd.merge( 
                            tempusCountsDF.loc[:, "gene_biotype"], 
                            elifeCountsDF, 
                            how='inner',
                            left_index=True,
                            right_index=True,
                            # left_on='gene_id', 
                            # right_on='gene', 
                            suffixes=('_t', '_e') )

elifeWithBiomarkerDF.index.name = "gene"

CPU times: user 111 ms, sys: 232 ms, total: 343 ms
Wall time: 703 ms


In [16]:
print(elifeWithBiomarkerDF.shape)

(76537, 225)


In [17]:
elifeWithBiomarkerDF.iloc[0:5, 0:5]

,gene_biotype,SRR14506659,SRR14506660,SRR14506661,SRR14506662
gene,,,,,
(A)n,Microsatellite,201.672053,110.450773,3722.776395,1394.605651
(AAA)n,Microsatellite,0.000000,0.000000,0.000000,0.000000
(AAAAAAC)n,Microsatellite,0.000000,0.000000,0.000000,0.000000
(AAAAAAG)n,Microsatellite,0.000000,0.000000,0.000000,0.000000
(AAAAAAT)n,Microsatellite,0.000000,0.000000,0.000000,0.000000


In [18]:
savePath = f'{dataOutDir}/annotated_elife_all_norm_counts_2023-05-18.csv'
elifeWithBiomarkerDF.to_csv( savePath )
print( f'saved:\n{savePath}')

saved:
/private/groups/kimlab/aedavids/elife/createNonCodingDataSet.out/data/annotated_elife_all_norm_counts_2023-05-18.csv


None


Index(['(A)n', '(AAA)n', '(AAAAAAC)n', '(AAAAAAG)n', '(AAAAAAT)n'], dtype='object')